# Support Vector Machines: exercises

This notebook is a structured exercise sheet to accompany the SVM demo.

The problems move from geometry to convex optimization, then to PCA and kernels.
For each exercise, first work it out on paper, then use the code cells to test your result.


In [4]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC, LinearSVC
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(7)
plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True

def plot_points(X, y, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots()
    for label, marker in [(-1, 'o'), (1, '^')]:
        ax.scatter(X[y == label, 0], X[y == label, 1], marker=marker, label=f'class {label}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    if title:
        ax.set_title(title)
    ax.legend()
    return ax


def _mesh_from_data(X, padding=1.0, n=400):
    x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
    y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, n), np.linspace(y_min, y_max, n))
    return xx, yy


def plot_decision_function(clf, X, y, ax=None, title=None, show_support=True):
    if ax is None:
        fig, ax = plt.subplots()
    plot_points(X, y, ax=ax)
    xx, yy = _mesh_from_data(X)
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.decision_function(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=['--', '-', '--'])
    if show_support and hasattr(clf, 'support_vectors_'):
        ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=180,
                   facecolors='none', edgecolors='k', linewidths=1.5, label='support vectors')
        ax.legend()
    if title:
        ax.set_title(title)
    return ax


def signed_distance(x, w, b):
    return (x @ w + b) / np.linalg.norm(w)


def fit_linear_svm(X, y, C=1e6):
    clf = SVC(kernel='linear', C=C)
    clf.fit(X, y)
    return clf


## Exercise 1. Distance to a hyperplane


Let the hyperplane be $H = \{x : w^\top x + b = 0\}$ with $w \neq 0$.

1. Show that the signed distance from a point $x_0$ to $H$ is

$$
\frac{w^\top x_0 + b}{\|w\|}.
$$

2. Deduce that the Euclidean distance is

$$
\frac{|w^\top x_0 + b|}{\|w\|}.
$$

Write your proof below, then verify numerically with a concrete example.

Chcemy znaleźć odległość punktu od płaszczyzny, będzie ona prostopadła do płaszczyzny.
$$
\hat n = \frac{w}{\|w\|}.
$$
Wektor $\hat n$ ma długość 1 i prostopadły do hiperpłaszczyzny. Weźmy dowolny punkt $x_H$ spełniający równanie $w^\top x_H + b = 0$.
Odległość $d(x_0, x_H)$ nie będzie prostopadła, będziemy szukać składowych wektora - $v=x_0 - x_H$. Bierzemy rzut wektora $v$ na kierunek normalny $\hat n$.

$$
\operatorname{dist}_{\pm}(x_0, H)=\hat n^\top(x_0 - x_H) = \frac{w^\top}{\|w\|}(x_0 - x_H) =\frac{w^\top x_0-w^\top x_H}{\|w\|}
$$
Wiemy, że punkt $x_H$ leży na płaszczyźnie $H$, stąd otrzymujemy: 

$$
\operatorname{dist}_{\pm}(x_0, H)=\frac{w^\top x_0 + b}{\|w\|}.
$$

Odległość euklidesowa wynosi:
$$
\operatorname{dist}(x_0, H)=\frac{|w^\top x_0 + b|}{\|w\|}.
$$

In [5]:
# Write a small numerical check here.
w = np.array([3.0, -4.0])
b = 2.0
x0 = np.array([1.0, 2.0])

# TODO: compute the signed distance
# długość wektora w
w_norm = np.linalg.norm(w)

# signed distance
signed_dist = (np.dot(w, x0) + b) / w_norm

# Euclidean distance
euclid_dist = abs(signed_dist)

print("w =", w)
print("b =", b)
print("x0 =", x0)
print(f"Signed distance = {signed_dist:.4f}")
print(f"Euclidean distance = {euclid_dist:.4f}")

w = [ 3. -4.]
b = 2.0
x0 = [1. 2.]
Signed distance = -0.6000
Euclidean distance = 0.6000


## Exercise 2. Margin width


Assume the classifier is normalized so that

$$
y_i(w^\top x_i+b) \ge 1.
$$

Show that the two margin hyperplanes are $w^\top x+b=1$ and $w^\top x+b=-1$, and prove that the distance between them is

$$
\frac{2}{\|w\|}.
$$

Then fit a nearly hard-margin linear SVM to a separable 2D dataset and verify the formula numerically.


In [ ]:
X = np.array([
    [-2.5, -0.8], [-2.0, 0.6], [-1.2, -1.3], [-0.9, 1.1],
    [ 1.1, -0.7], [ 1.5, 1.4], [ 2.0, -1.0], [ 2.6, 0.9],
])
y = np.array([-1, -1, -1, -1, 1, 1, 1, 1])

# TODO: fit a linear SVM with large C and compute 2 / ||w||


## Exercise 3. Convexity of hard-margin SVM


Consider the optimization problem

$$
\min_{w,b} \frac12\|w\|^2
\quad\text{subject to}\quad
y_i(w^\top x_i+b) \ge 1.
$$

Explain carefully why this is a convex optimization problem.

Your answer should explicitly identify:

- the objective function,
- the feasible set,
- why local minima are global minima in this setting.


In [ ]:
# Write a short paragraph in markdown above.


## Exercise 4. Dual representation of $w$


Starting from the Lagrangian

$$
L(w,b,\alpha) = \frac12\|w\|^2 - \sum_i \alpha_i\bigl(y_i(w^\top x_i+b)-1\bigr),
\qquad \alpha_i \ge 0,
$$

show that stationarity implies

$$
w = \sum_i \alpha_i y_i x_i,
\qquad
\sum_i \alpha_i y_i = 0.
$$

Then, using a fitted linear SVM, reconstruct $w$ from the support vectors and compare it with the model coefficient.


In [ ]:
X = np.array([
    [-2.5, -0.8], [-2.0, 0.6], [-1.2, -1.3], [-0.9, 1.1],
    [ 1.1, -0.7], [ 1.5, 1.4], [ 2.0, -1.0], [ 2.6, 0.9],
])
y = np.array([-1, -1, -1, -1, 1, 1, 1, 1])
clf = SVC(kernel='linear', C=1e6).fit(X, y)

# TODO: extract dual coefficients and reconstruct w


## Exercise 5. Effect of the soft-margin parameter $C$


Generate an overlapping binary dataset and fit three linear SVMs with different values of $C$.

1. Plot the decision boundary and margins.
2. Compare the number of support vectors.
3. Report the training accuracy and margin width.
4. Explain in words how increasing $C$ changes the fit.


In [ ]:
X, y = make_blobs(n_samples=100, centers=[(-1.0, 0.0), (1.0, 0.0)],
                  cluster_std=[1.2, 1.2], random_state=5)
y = 2 * y - 1
Cs = [0.1, 1.0, 100.0]

# TODO: fit and compare the three models


## Exercise 6. PCA versus SVM


Construct or simulate a 2D dataset for which:

- the first principal component is a poor direction for class separation,
- but a linear SVM still separates the classes well.

Then:

1. plot the data,
2. overlay the first principal component,
3. fit a linear SVM,
4. compare classification accuracy before and after projection to the first principal component.


In [ ]:
# Hint: create vertically elongated classes whose centers differ mostly in x.


## Exercise 7. Kernel trick on a nonlinear dataset


Use either the moons dataset or the circles dataset.

1. Fit a linear SVM.
2. Fit a polynomial-kernel SVM.
3. Fit an RBF-kernel SVM.
4. Plot the decision boundaries and explain which kernel works best and why.


In [ ]:
X, y = make_moons(n_samples=200, noise=0.18, random_state=2)
y = 2 * y - 1

# TODO: compare linear, polynomial, and RBF kernels


## Exercise 8. Short conceptual comparison


Answer the following in complete sentences.

1. Why is PCA unsupervised while SVM is supervised?
2. Why does the dual formulation naturally lead to kernels?
3. Why are support vectors the only training points that matter directly in the final decision function?


In [ ]:
# Write your answers in markdown above.
